In [ ]:
%load_ext autoreload
%autoreload all

In [ ]:
# Standard Library imports
from pathlib import Path
from datetime import datetime
from tqdm import trange

# External imports
import cv2
import supervision as sv

In [ ]:
ROOT_PATH = Path(<FILL_ME>)

In [ ]:
yolo_dirpath = str(ROOT_PATH / "dataset_yolo")

ds_train = sv.DetectionDataset.from_yolo(
    images_directory_path=f"{yolo_dirpath}/images/train",
    annotations_directory_path=f"{yolo_dirpath}/labels/train",
    data_yaml_path=f"{yolo_dirpath}/data.yaml",
)

In [ ]:
ds_train.image_paths = sorted(ds_train.image_paths)

In [ ]:
ds_train.classes

In [ ]:
len(ds_train)

### Visualization of frames

In [ ]:
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

start_image_idx = 0
num_images = 1
end_image_idx = start_image_idx + num_images
for i in range(start_image_idx, end_image_idx):
    filepath, image, annotations = ds_train[i]

    labels = [ds_train.classes[class_id] for class_id in annotations.class_id]

    annotated_image = image.copy()
    annotated_image = box_annotator.annotate(annotated_image, annotations)
    annotated_image = label_annotator.annotate(annotated_image, annotations, labels)
    print(filepath, annotations)
    sv.plot_image(annotated_image, size=(14, 10))

### Video composition
Generate a video from the exported YOLO-annotated frames.

In [ ]:
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

start_image_idx = 0
num_images = len(ds_train)
end_image_idx = start_image_idx + num_images
fps = 2
output_path = ROOT_PATH / f"annotated_video-exported-on-{datetime.now().strftime('%Y%m%d_%H%M%S')}.mp4"

filepath, image, annotations = ds_train[start_image_idx]
h, w = image.shape[:2]

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
video = cv2.VideoWriter(str(output_path), fourcc, fps, (w, h), isColor=True)


for i in trange(start_image_idx, end_image_idx):
    filepath, image, annotations = ds_train[i]
    labels = [ds_train.classes[class_id] for class_id in annotations.class_id]

    annotated_image = image.copy()
    annotated_image = box_annotator.annotate(annotated_image, annotations)
    annotated_image = label_annotator.annotate(annotated_image, annotations, labels)

    video.write(annotated_image)

video.release()
print(f"Video written to '{output_path.resolve()}'")